In [0]:
WITH audit AS (
  SELECT
    api_access,
    COUNT(*) AS row_count,

    ROUND(SUM(cost_eur), 2) AS total_cost_eur,
    ROUND(SUM(cost_usd), 2) AS total_cost_usd,

    SUM(CASE
          WHEN application IS NULL OR TRIM(application) = ''
          THEN 1 ELSE 0
        END) AS missing_application,

    SUM(CASE
          WHEN service IS NULL OR TRIM(service) = ''
          THEN 1 ELSE 0
        END) AS missing_service,

    SUM(CASE
          WHEN model IS NULL OR TRIM(model) = ''
          THEN 1 ELSE 0
        END) AS missing_model,

    SUM(CASE
          WHEN spoke IS NULL OR TRIM(spoke) = ''
          THEN 1 ELSE 0
        END) AS missing_spoke,

    SUM(CASE
          WHEN cost_eur IS NULL
          THEN 1 ELSE 0
        END) AS missing_eur,

    SUM(CASE
          WHEN cost_usd IS NULL
          THEN 1 ELSE 0
        END) AS missing_usd,

    SUM(CASE
          WHEN application_inferred = TRUE
          THEN 1 ELSE 0
        END) AS application_inferred_rows,

    SUM(CASE
          WHEN spoke_inferred = TRUE
          THEN 1 ELSE 0
        END) AS spoke_inferred_rows,

    SUM(CASE
          WHEN model_inferred = TRUE
          THEN 1 ELSE 0
        END) AS model_inferred_rows

  FROM pierre_delaguiche_d.default.ai_foundry_costs_enriched
  GROUP BY api_access
)

SELECT
  api_access,
  row_count,
  total_cost_eur,
  total_cost_usd,
  missing_application,
  missing_service,
  missing_model,
  missing_spoke,
  missing_eur,
  missing_usd,
  application_inferred_rows,
  spoke_inferred_rows,
  model_inferred_rows,

  ROUND(
    100.0 * (row_count - missing_application) / row_count,
    2
  ) AS application_coverage_pct,

  ROUND(
    100.0 * (row_count - missing_service) / row_count,
    2
  ) AS service_coverage_pct,

  ROUND(
    100.0 * (row_count - missing_model) / row_count,
    2
  ) AS model_coverage_pct,

  ROUND(
    100.0 * (row_count - missing_spoke) / row_count,
    2
  ) AS spoke_coverage_pct

FROM audit
ORDER BY total_cost_eur DESC;